In [1]:
from pathlib import Path
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class CharCorpusDataset(Dataset):
    def __init__(self, text: str, seq_len: int = 64):
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.stoi = {ch: idx for idx, ch in enumerate(self.chars)}
        self.itos = {idx: ch for idx, ch in enumerate(self.chars)}
        self.seq_len = seq_len

        encoded = [self.stoi[ch] for ch in text]
        self.data = torch.tensor(encoded, dtype=torch.long)

    def __len__(self) -> int:
        return max(0, len(self.data) - self.seq_len - 1)

    def __getitem__(self, idx: int):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


class CharRNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        hidden_dim: int = 256,
        num_layers: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: torch.Tensor, hidden: torch.Tensor | None = None):
        emb = self.embedding(x)
        out, hidden = self.rnn(emb, hidden)
        logits = self.fc(out)
        return logits, hidden


@torch.no_grad()
def generate_text(
    model: CharRNN,
    dataset: CharCorpusDataset,
    start_text: str = "The ",
    max_new_chars: int = 250,
    temperature: float = 0.8,
    device: torch.device = torch.device("cpu"),
) -> str:
    model.eval()
    # Fall back safely if start_text contains unseen characters
    fallback_idx = 0
    input_ids = [dataset.stoi.get(ch, fallback_idx) for ch in start_text]
    if not input_ids:
        input_ids = [fallback_idx]

    x = torch.tensor([input_ids], dtype=torch.long, device=device)
    logits, hidden = model(x)

    generated = list(start_text)
    # Sample first next token from the last step of the prompt
    next_logits = logits[:, -1, :] / max(temperature, 1e-5)
    probs = torch.softmax(next_logits, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1).item()
    generated.append(dataset.itos[next_id])

    cur_input = torch.tensor([[next_id]], dtype=torch.long, device=device)
    for _ in range(max_new_chars - 1):
        out_logits, hidden = model(cur_input, hidden)
        step_logits = out_logits[:, -1, :] / max(temperature, 1e-5)
        step_probs = torch.softmax(step_logits, dim=-1)
        next_id = torch.multinomial(step_probs, num_samples=1).item()
        generated.append(dataset.itos[next_id])
        cur_input = torch.tensor([[next_id]], dtype=torch.long, device=device)

    return "".join(generated)


def train_text_rnn(
    corpus_text: str,
    seq_len: int = 64,
    batch_size: int = 64,
    epochs: int = 10,
    lr: float = 2e-3,
    save_path: Path = Path("outputs/char_rnn.pt"),
) -> None:
    start_time = time.perf_counter()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        torch.set_num_threads(max(1, min(4, torch.get_num_threads())))

    dataset = CharCorpusDataset(corpus_text, seq_len=seq_len)
    if len(dataset) == 0:
        raise ValueError(f"Corpus is too short ({len(corpus_text)} chars) for seq_len={seq_len}.")

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    model = CharRNN(vocab_size=dataset.vocab_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    save_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Device: {device} | Vocab size: {dataset.vocab_size} | Sequences: {len(dataset)}")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        steps = 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits, _ = model(x)
            # Reshape (batch, seq_len, vocab) -> (batch * seq_len, vocab)
            loss = criterion(logits.reshape(-1, dataset.vocab_size), y.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            steps += 1

        avg_loss = total_loss / max(steps, 1)
        elapsed = time.perf_counter() - start_time
        print(f"Epoch {epoch}/{epochs} ({elapsed:.1f}s) | Loss: {avg_loss:.4f}")

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "chars": dataset.chars,
            "seq_len": seq_len,
        },
        save_path,
    )
    print(f"Saved model to {save_path}")

    sample = generate_text(model, dataset, start_text="The ", max_new_chars=200, temperature=0.8, device=device)
    print("\n--- Generated Sample ---")
    print(sample)


if __name__ == "__main__":
    sample_corpus = (
        "The cat sat on the mat while the dog watched from the porch. "
        "The sun set over the quiet street and the wind moved through the trees. "
        "Every morning the cat waited by the door and the dog barked at the breeze. "
    ) * 40

    train_text_rnn(
        corpus_text=sample_corpus,
        seq_len=48,
        batch_size=32,
        epochs=15,
        lr=2e-3,
    )

Device: cpu | Vocab size: 28 | Sequences: 8271
Epoch 1/15 (58.6s) | Loss: 0.2671
Epoch 2/15 (108.6s) | Loss: 0.0602
Epoch 3/15 (157.5s) | Loss: 0.0583
Epoch 4/15 (207.5s) | Loss: 0.0578
Epoch 5/15 (257.5s) | Loss: 0.0574
Epoch 6/15 (306.0s) | Loss: 0.0572
Epoch 7/15 (355.4s) | Loss: 0.0568
Epoch 8/15 (405.0s) | Loss: 0.0567
Epoch 9/15 (454.5s) | Loss: 0.0568
Epoch 10/15 (502.9s) | Loss: 0.0566
Epoch 11/15 (552.8s) | Loss: 0.0566
Epoch 12/15 (602.4s) | Loss: 0.0565
Epoch 13/15 (650.5s) | Loss: 0.0565
Epoch 14/15 (699.6s) | Loss: 0.0565
Epoch 15/15 (749.0s) | Loss: 0.0566
Saved model to outputs/char_rnn.pt

--- Generated Sample ---
The cat sat on the mat while the dog watched from the porch. The sun set over the quiet street and the wind moved through the trees. Every morning the cat waited by the door and the dog barked at the bree
